# PyIBS Example 2: Maximum-likelihood estimation with PyBADS

Fit a model using only its simulator: this notebook passes IBS estimates of a negative log-likelihood to PyBADS, then checks the fitted parameters using a closed-form likelihood.

We use the orientation discrimination model and data set introduced in the [first notebook](./pyibs_example_1_basic_usage.ipynb). This is the second of three examples; the [third](./pyibs_example_3_posterior_with_pyvbmc.ipynb) performs Bayesian inference with PyVBMC. The code is also available as a [Python script](https://github.com/acerbilab/pyibs/blob/main/examples/scripts/pyibs_example_2_maximum_likelihood_with_pybads.py).

See the [installation guide](https://acerbilab.github.io/pyibs/installation.html) for PyIBS and the packages needed to run this notebook.

In [1]:
import math

import numpy as np
from pybads import BADS
from scipy.optimize import minimize

from pyibs import IBS
from pyibs.examples.psycho_model import psycho_generator, psycho_neg_logl

## 0. Choosing an optimizer for a noisy likelihood

Repeated IBS evaluations at the same parameters give different values. An optimizer needs to distinguish that sampling noise from a useful improvement. Bayesian Adaptive Direct Search (BADS) supports noisy objectives; its Python implementation, PyBADS, can use the estimated SD of each evaluation when the target supplies it.

PyBADS and PyVBMC, used in the [next notebook](./pyibs_example_3_posterior_with_pyvbmc.ipynb), are among the lab's [tools for fitting models to data](https://acerbilab.org/model-fitting/).

## 1. The model and its search region

The parameter vector is $\theta=(\eta,\mu,\gamma)$: log sensory-noise SD, response bias and lapse rate. We generate the same 600 trials as in the first notebook: normally distributed orientations of SD 3°, and responses at $\theta_\text{true}=(\log 1,0.2,0.03)$.

Seeds spawned from a common root give separate streams for data generation, IBS, the starting point and PyBADS. The data stream is shared across this series; the other streams keep sampling and optimization reproducible.

In [2]:
# Independent seeds for the data, the IBS estimates, the start and PyBADS
seed_data, seed_ibs, seed_start, seed_bads = np.random.SeedSequence(
    2026
).spawn(4)

theta_true = np.array([math.log(1.0), 0.2, 0.03])  # log(sigma), bias, lapse
n_trials = 600

rng = np.random.default_rng(seed_data)
S = 3 * rng.standard_normal((n_trials, 1))  # Orientation of each trial
R = psycho_generator(theta_true, S, rng)  # Response of each trial

The bounds have two roles:

- Hard bounds `LB` and `UB` define where PyBADS may evaluate the model. The lapse rate is at least 0.01, so either response has probability at least 0.005 on every trial. This limits the cost of IBS: a near-impossible response would otherwise require many samples before a match.
- Plausible bounds `PLB` and `PUB` describe a likely region for the solution. PyBADS uses that box to place initial points and scale the search.

These bounds and the data-generation recipe follow MATLAB IBS's `ibs_example.m`.

In [3]:
LB = np.array([math.log(0.1), -2.0, 0.01])  # Lower bounds
UB = np.array([math.log(10.0), 2.0, 1.0])  # Upper bounds
PLB = np.array([math.log(0.2), -1.0, 0.02])  # Plausible lower bounds
PUB = np.array([math.log(5.0), 1.0, 0.2])  # Plausible upper bounds

For a reference fit, we minimize the closed-form negative log-likelihood with L-BFGS-B. This target is deterministic and smooth. Its numerical minimum gives us a benchmark for the later fit with IBS.

In [4]:
exact_fit = minimize(
    psycho_neg_logl,
    (PLB + PUB) / 2,
    args=(S, R),
    method="L-BFGS-B",
    bounds=list(zip(LB, UB)),
)
theta_ml, neg_logl_min = exact_fit.x, exact_fit.fun
print(f"Reference maximum-likelihood point: {theta_ml.round(4)}")
print(f"Reference minimum of the negative log-likelihood: {neg_logl_min:.2f}")

Reference maximum-likelihood point: [0.0584 0.2828 0.0184]
Reference minimum of the negative log-likelihood: 151.78


## 2. Defining the IBS target

PyBADS minimizes a target value. We return the *negative* log-likelihood estimate and its estimated SD as the tuple `(value, sd)`, using `additional_output="std"` and `options={"specify_target_noise": True}`.

The second component is measured in the same units as the target value. For example, variance 9 corresponds to SD 3; passing the variance would give PyBADS the wrong noise scale.

We use 100 repeats, which give an SD near 1 for this data set. PyBADS works best with noise of about 1 or less near the solution, though larger noise farther away is usually acceptable (see the [PyBADS FAQ](https://acerbilab.github.io/pybads/faq.html#faq-can-pybads-handle-any-arbitrary-amount-of-noise-in-the-objective)).

PyBADS requires a finite, positive SD. With the unit weights used here, IBS reports zero when every trial matches on its first sample in every repeat. That is extremely unlikely for this data set, but IBS warns if it happens. The [PyIBS FAQ](https://acerbilab.github.io/pyibs/faq.html#faq-why-is-the-sd-of-the-estimate-zero-and-why-do-pybads-and-pyvbmc-refuse-it) explains how to handle it.

In [5]:
ibs = IBS(psycho_generator, R, S, vectorized=True, random_seed=seed_ibs)


def target(theta):
    """IBS estimate of the negative log-likelihood, and its SD."""
    return ibs(theta, num_reps=100, additional_output="std")

We draw a starting point uniformly within the plausible box, as `ibs_example.m` does, and compare one IBS evaluation there with the closed-form value. This gives an initial check that the target has the intended sign and noise scale.

In [6]:
x0 = PLB + np.random.default_rng(seed_start).random(3) * (PUB - PLB)
value, sd = target(x0)
print(f"Starting point: {x0.round(4)}")
print(f"Target at the start: {value:.2f} +/- {sd:.2f}")
print(f"Exact value:         {psycho_neg_logl(x0, S, R):.2f}")

Starting point: [ 0.1637 -0.5321  0.1351]
Target at the start: 192.54 +/- 1.24
Exact value:         194.46


## 3. Running the optimization

The `BADS` object receives the target, starting point and bounds. `specify_target_noise` tells it to read the target's `(value, sd)` output; `random_seed` fixes PyBADS's own random stream. The separate seed of the `IBS` object controls the simulator draws.

`optimize()` prints progress as it searches. It may also print a documentation tip; setting `options["show_tips"] = False` disables tips.

In [7]:
bads = BADS(
    target,
    x0,
    LB,
    UB,
    PLB,
    PUB,
    options={"specify_target_noise": True, "random_seed": seed_bads},
)
optimize_result = bads.optimize()

Variables (index) internally transformed to log coordinates: [2].
Beginning optimization of a STOCHASTIC objective function (specified noise)

Tip: If you work with a coding agent, give it the PyBADS skill from GitHub, or copy the skills/pybads folder of PyBADS's repository into the agent's skill directory: it points the agent to the parts of PyBADS's documentation relevant to your task.
https://github.com/acerbilab/pybads/blob/main/skills/pybads/SKILL.md

 Iteration    f-count      E[f(x)]        SD[f(x)]           MeshScale          Method              Actions
     0           1         194.397         1.24537               1                                  
     0          33         156.692         1.24537               1          Initial mesh            Initial points
     1          38         156.692         1.10304             0.5          Refine grid             Train
     2          39         153.892         1.04331             0.5      Successful search (ES-wcm)        
  

## 4. Assessing the solution

`optimize_result["x"]` contains the fitted parameters. `fval` is the precision-weighted mean of PyBADS's final target evaluations there, and `fsd` is the standard error of that mean. We compare the parameters with the generating vector and the reference fit.

In [8]:
theta_bads = optimize_result["x"]
fval, fsd = optimize_result["fval"], optimize_result["fsd"]
print(f"PyBADS estimate at its solution: {fval:.2f} +/- {fsd:.2f}")
print(f"Target evaluations: {optimize_result['func_count']}")
print()
print(f"{'':17}{'eta':>8}{'bias':>8}{'lapse':>8}")
for name, theta in [
    ("Generating", theta_true),
    ("Reference ML", theta_ml),
    ("PyBADS solution", theta_bads),
]:
    print(f"{name:17}" + "".join(f"{x:8.4f}" for x in theta))

PyBADS estimate at its solution: 151.87 +/- 0.33
Target evaluations: 343

                      eta    bias   lapse
Generating         0.0000  0.2000  0.0300
Reference ML       0.0584  0.2828  0.0184
PyBADS solution    0.0448  0.2634  0.0242


A fresh, more precise evaluation helps assess a solution chosen using noisy values. We use 1,000 IBS repeats, ten times the number used during the search, as `ibs_example.m` does, and compare that estimate with the closed-form likelihood at the fitted parameters. We also measure how far the fitted value lies above the reference minimum.

In [9]:
neg_logl, neg_logl_sd = ibs(theta_bads, num_reps=1000, additional_output="std")
neg_logl_exact = psycho_neg_logl(theta_bads, S, R)
print(
    "IBS estimate at the solution (1,000 repeats): "
    f"{neg_logl:.2f} +/- {neg_logl_sd:.2f}"
)
print(f"Exact value at the solution:  {neg_logl_exact:.2f}")
print(f"Reference minimum:            {neg_logl_min:.2f}")
print(f"Difference:                   {neg_logl_exact - neg_logl_min:.2f}")

IBS estimate at the solution (1,000 repeats): 152.43 +/- 0.33
Exact value at the solution:  151.89
Reference minimum:            151.78
Difference:                   0.11


The 1,000-repeat estimate lies within two of its SDs of the exact value at the solution, and the fitted negative log-likelihood is close to the reference minimum relative to the roughly unit noise of each search evaluation. This run illustrates that a noisy target can still locate a useful solution; the closed form lets us check the fit independently.

The reference maximum-likelihood point differs from the generating parameters because a finite data set has sampling variation; for a well-identified model under the assumed data-generating process, that variation decreases with more trials. The PyBADS solution also differs from the reference point, since PyBADS searched a noisy target.

For a model whose likelihood cannot be checked directly, run PyBADS from several starting points and compare the solutions with fresh, more precise evaluations. The [PyBADS FAQ](https://acerbilab.github.io/pybads/faq.html#faq-how-do-i-run-pybads-from-several-starting-points) describes this workflow.

## 5. Moving to Bayesian inference

The IBS target supplies both a negative log-likelihood estimate and its uncertainty. PyBADS uses those evaluations to search for maximum-likelihood parameters. The [next notebook](./pyibs_example_3_posterior_with_pyvbmc.ipynb) uses the same simulator to infer a distribution over parameters and estimate the model evidence with PyVBMC.

## References

1. van Opheusden, B., Acerbi, L. & Ma, W. J. (2020). Unbiased and efficient log-likelihood estimation with inverse binomial sampling. *PLOS Computational Biology* 16(12): e1008483. [https://doi.org/10.1371/journal.pcbi.1008483](https://doi.org/10.1371/journal.pcbi.1008483)
2. Singh, G. S. & Acerbi, L. (2024). PyBADS: Fast and robust black-box optimization in Python. *Journal of Open Source Software* 9(94): 5694. [https://doi.org/10.21105/joss.05694](https://doi.org/10.21105/joss.05694)
3. Acerbi, L. & Ma, W. J. (2017). Practical Bayesian optimization for model fitting with Bayesian adaptive direct search. In *Advances in Neural Information Processing Systems 30*: 1834-1844. [https://arxiv.org/abs/1705.04405](https://arxiv.org/abs/1705.04405)